In [1]:
from pathlib import Path
import json
from typing import List, Union, Dict

import soundfile as sf
import torchaudio
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import torch

from mm_story_agent.prompts_en import story_to_music_reviser_system, story_to_music_reviewer_system
from mm_story_agent.base import register_tool, init_tool_instance




c:\Users\rajad\anaconda3\envs\story\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing ToolRegistry
Importing all tools...
Registering QAOutlineStoryWriter...
Registering qa_outline_story_writer: QAOutlineStoryWriter
Registering tool: qa_outline_story_writer
Registering LLM agents...
Registering gemini: GeminiAgent
Registering tool: gemini
Registering openai: OpenAIAgent
Registering tool: openai
Registering musicgen_t2m: MusicGenAgent
Registering tool: musicgen_t2m
Registering audioldm2_t2a: AudioLDM2Agent
Registering tool: audioldm2_t2a
Registering gtts: GoogleTTSAgent
Registering tool: gtts
Registering elevenlabs: ElevenLabsAgent
Registering tool: elevenlabs
Registering story_diffusion_t2i: StoryDiffusionAgent
Registering tool: story_diffusion_t2i
Registering freesound_sfx_retrieval: FreesoundSfxAgent
Registering tool: freesound_sfx_retrieval
Registering freesound_music_retrieval: FreesoundMusicAgent
Registering tool: freesound_music_retrieval
Using ImageMagick from: C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe
Registering slideshow_video_compose

In [3]:
class MusicGenSynthesizer:

    def __init__(self,
                 model_name: str = 'facebook/musicgen-stereo-large',
                 device: str = 'cpu',
                 sample_rate: int = 16000,
                 ) -> None:
        self.device = device
        self.model = MusicgenForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            attn_implementation="eager"
        ).to(device)
        
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.sample_rate = sample_rate
    
    def call(self,
             prompt: Union[str, List[str]],
             save_path: Union[str, Path],
             duration: float = 30.0,
             ):
        inputs = self.processor(
            text=[prompt],
            padding=True,
            return_tensors="pt",
        ).to(self.device)
        seq_length = int(51.2 * duration)
        wav = self.model.generate(**inputs, max_new_tokens=seq_length)[0, 0].cpu()
        wav = torchaudio.functional.resample(wav, self.model.config.audio_encoder.sampling_rate, self.sample_rate)
        sf.write(save_path, wav.numpy(), self.sample_rate)

In [4]:
music_prompt = "A light and whimsical opening, perhaps with flute and harp, to evoke a sense of peaceful nature. This shifts to a more scheming and slightly darker tone with bassoon and pizzicato strings as the heron's plan unfolds. Suspenseful music with increasing tempo and intensity as the heron carries out his deception. The music turns dramatic and urgent with sharp string accents when Bholu discovers the truth and attacks the heron. Finally, a triumphant and joyful fanfare with full orchestra as the crab is celebrated."
save_path = Path("C:\\Users\\rajad\\AptSmart\\story\\MM_StoryAgent\\test_music")

In [ ]:
generation_agent = MusicGenSynthesizer(
            model_name= 'facebook/musicgen-stereo-large',
            device="cpu",
            sample_rate=16000,
        )
generation_agent.call(
            prompt=music_prompt,
            save_path=save_path / "music.wav",
            duration=30.0,
        )

Fetching 2 files: 100%|██████████| 2/2 [06:29<00:00, 194.91s/it]
Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_r